In [ ]:
import os 
import base64
from openai import AzureOpenAI
import json
import re
import numpy as np
import datetime
import pandas as pd
import time
import ast
from collections import defaultdict

In [ ]:
endpoint = "<your_azure_endpoint_here>"
model_name = "gpt-4o"
deployment = "gpt-4o"

subscription_key = "<your_subscription_key_here>"
api_version = "<your_api_version_here>"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

In [ ]:
df_note = pd.read_csv("note_data.csv", encoding='utf-8')
df_note.reset_index(drop=True, inplace=True)

In [ ]:

all_outputs = []

for i, note in enumerate(df_note["note_text"]):
    system_message = """You are a medical expert. Your task is to analyze a medical note and extract referral information in a structured JSON format. Do not include any explanations, disclaimers, or text outside of the JSON object."""
    
    prompt = f"""
    Task:
    Using ONLY the MEDICAL NOTE {note}, identify clinical actions that are explicitly documented as being initiated, ordered, or performed in response to genetic findings or a Genome-informed Risk Assessment (GIRA).

    Definition:
    A GIRA-informed clinical action is one that the MEDICAL NOTE explicitly links to:
    - a genetic result, variant, mutation, polygenic risk, calculated risk (e.g., BOADICEA), 
    – a documented Genome-Informed Risk Assessment or interpretation.

    Rules:
    –	The MEDICAL NOTE is the ONLY evidence that an action occurred.
    –	The MEDICAL NOTE must explicitly indicate a genetic or GIRA-related reason for the action.
    –	Do NOT infer causality, assume intent, or recommend actions.
    –	Only consider genetic results related to the following conditions:
    -	Asthma
    -	Atrial fibrillation
    -	Breast cancer
    -	Chronic kidney disease
    -	Coronary heart disease
    -	Hypercholesterolemia
    -	Obesity / BMI
    -	Prostate cancer
    -	Type 1 diabetes
    -	Type 2 diabetes

    Action Types (GIRA-Aligned):
    –	referral
    –	lab_test
    –	imaging_or_monitoring

    Genetic-Related Specialist Referrals:
    Only classify an action as a referral if the MEDICAL NOTE explicitly documents referral to one of the following AND links it to a genetic or GIRA-related reason:

    –	Oncologist
    –	Surgeon (oncology-related)
    –	Cardiologist
    –	Electrophysiologist (cardiology)
    –	Lipids Specialist (cardiology)
    –	Nephrologist
    –	Endocrinologist
    –	Gynecologist
    –	Urologist
    –	Dietician
    –	Nutritionist
    –	Genetic Counselor

    Extraction:
    For each GIRA-informed action explicitly documented, extract:

    –	action_type
    –	action_name (e.g., cardiology referral, lipid panel, ECG, renal ultrasound)
    –	evidence_citation (verbatim text demonstrating both the action and its genetic/GIRA-informed reason)

    Output (JSON Only):

{
  "actions_present": true or false,
  "actions": [
    {
      "action_type": string,
      "action_name": string,
      "evidence_citation": string
    }
    ...
    /* repeat for additional clinical actions if applicable */
  ]
}
"""

    chat_promp = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_message
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    try:
        completion = client.chat.completions.create(
            model=deployment,
            messages=chat_promp,
            max_tokens=4096,
            temperature=0,
            frequency_penalty=0,
            presence_penalty=0,
            stream=False
        )

        content = completion.choices[0].message.content.strip()
        
        if content.startswith("```") and content.endswith("```"):
            content = content[3:-3].strip()
        
        if content.startswith("json"):
            content = content[4:].strip()

        try:
            parsed = json.loads(content)
            parsed["person"] = int(df_note["MRN"][i])
            parsed["note_date"] = str(df_note["note_date"][i])
            parsed["note_id"] = int(df_note["note_id"][i])
            parsed["note_title"] = str(df_note["note_title"][i])
            parsed["GIRA_date"] = str(df_note["GIRA_date"][i])
            all_outputs.append(parsed)
        except json.JSONDecodeError:
            print(f"Warning: Could not parse JSON for index {i}. Content was:\n{content_clean}")
            all_outputs.append({"error": "Invalid JSON", "index": i, "raw": content_clean})

    except Exception as e:
        print(f"Error during inference at index {i}: {e}")
        all_outputs.append({"error": "Model error", "index": i, "exception": str(e)})

with open("GPT4o_baseline.json", "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, ensure_ascii=False, indent=2)


In [ ]:
def safe_json_loads(raw_str):
    try:
        return json.loads(raw_str)
    except json.JSONDecodeError:
        try:
            fixed = raw_str.replace("\\\'", "'") 
            return json.loads(fixed)
        except Exception as e2:
            return None

with open('GPT4o_baseline.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

rows = []
for entry in data:
    parsed = None

    if 'error' in entry and 'raw' in entry:
        parsed = safe_json_loads(entry['raw'])
        index = entry.get("index")
        if parsed and parsed.get("referred") == True:
            entry = parsed
            person = int(df_note.loc[index, "MRN"])
            note_date = str(df_note.loc[index, "note_date"])
            GIRA_date = str(df_note.loc[index, "GIRA_date"])
            note_title = str(df_note.loc[index, "note_title"])
            note_id = int(df_note.loc[index, "note_id"])
        else:
            print(f"Failed to parse index: {index}")

            continue
            
    elif entry.get("referred") == True:
        parsed = entry
        person = parsed.get("person")
        note_date = parsed.get("note_date")
        GIRA_date = parsed.get("GIRA_date")
        note_title = parsed.get("note_title")
        note_id = parsed.get("note_id")
    else:
        continue

    specialists = parsed.get("specialists", [])
    for specialist in specialists:
        row = {
            "person": person,
            "GIRA_date": GIRA_date,
            "note_date": note_date,
            "note_id": note_id,
            "note_title": note_title,
            "specialist_type": specialist.get("specialist_type"),
            "referral_confirmation_citation": specialist.get("referral_confirmation_citation"),
            "specialist_type_citation": specialist.get("specialist_type_citation")
        }
        rows.append(row)
        
df = pd.DataFrame(rows)
